In [ ]:
# ============================================================
# CELL 1: GPU Check & Environment Setup
# ============================================================
import torch
import os

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Total VRAM   : {total_mem:.2f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️  No GPU found — using CPU (training will be slow)")
    DEVICE = torch.device("cpu")

print(f"\nUsing device: {DEVICE}")

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os, random, shutil, warnings, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print("✅ All imports successful.")

In [ ]:
# ============================================================
# CELL 3: Configuration
# ============================================================

# ── Paths ──────────────────────────────────────────────────
DATASET_PATH   = Path("/kaggle/input/datasets/bapdesilva/betel-leaf-quality-dataset-2025/betel-quality-ds")
AUGMENTED_PATH = Path("/kaggle/working/augmented_dataset")
FINAL_PATH     = Path("/kaggle/working/final_dataset")

# ── Dataset ────────────────────────────────────────────────
TARGET_PER_CLASS = 2500
VAL_SPLIT        = 0.20
TEST_SPLIT       = 0.05 
IMG_SIZE         = 224

# ── Training hyper-parameters ──────────────────────────────
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR_HEAD      = 1e-3
LR_FINE      = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 7
UNFREEZE_PCT = 0.30

CLASS_NAMES = [
    "Grade A Quality",
    "Grade B Quality",
    "Grade C Quality",
    "Grade D Quality",
    "Grade E Quality",
]
NUM_CLASSES = len(CLASS_NAMES)

print("✅ Configuration set.")
print(f"   Dataset path  : {DATASET_PATH}")
print(f"   Target/class  : {TARGET_PER_CLASS}")
print(f"   Val split     : {VAL_SPLIT*100:.0f}%")
print(f"   test split     : {TEST_SPLIT*100:.0f}%")
print(f"   Num classes   : {NUM_CLASSES}")
print(f"   Device        : {DEVICE}")

In [ ]:
# ============================================================
# CELL 4: Inspect Dataset Structure
# ============================================================
print("Dataset folder structure:")
print("=" * 50)

found_classes = []
for item in sorted(DATASET_PATH.iterdir()):
    if item.is_dir():
        exts  = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
        count = len([f for f in item.iterdir()
                     if f.suffix.lower() in exts])
        print(f"  [{item.name}]  →  {count} images")
        found_classes.append(item.name)

print(f"\nFound {len(found_classes)} class folders:")
for c in found_classes:
    print(f"   {c}")

# Auto-update CLASS_NAMES if folder names differ slightly
if set(found_classes) != set(CLASS_NAMES):
    print("\n⚠️  Folder names differ from CLASS_NAMES config.")
    print("   Updating CLASS_NAMES to match actual folders...")
    CLASS_NAMES = sorted(found_classes)
    NUM_CLASSES = len(CLASS_NAMES)
    print(f"   Updated CLASS_NAMES: {CLASS_NAMES}")

In [ ]:
# ============================================================
# CELL 5: Offline Augmentation (run ONCE — before training)
# ============================================================

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                            scale=(0.85, 1.15), shear=10),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

def augment_class_folder(src_folder: Path, dst_folder: Path, target: int):
    dst_folder.mkdir(parents=True, exist_ok=True)
    exts     = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
    originals = [p for p in src_folder.iterdir()
                 if p.suffix.lower() in exts]

    if not originals:
        print(f"  ⚠️  No images found in {src_folder} — skipping.")
        return 0

    count = 0

    # 1. Copy originals
    for p in originals:
        shutil.copy(p, dst_folder / p.name)
        count += 1

    # 2. Augment until target reached
    idx = 0
    while count < target:
        src_img = originals[idx % len(originals)]
        img     = Image.open(src_img).convert("RGB")
        aug_img = aug_transform(img)
        aug_img.save(dst_folder / f"aug_{count:05d}.jpg")
        count += 1
        idx   += 1

    return count


print("=" * 55)
print(f"OFFLINE AUGMENTATION  (target = {TARGET_PER_CLASS} images/class)")
print("=" * 55)

total_generated = 0
for cls in CLASS_NAMES:
    src = DATASET_PATH / cls
    dst = AUGMENTED_PATH / cls

    if not src.exists():
        print(f"  ❌ Folder not found: {src}")
        continue

    n_orig = len([f for f in src.iterdir()
                  if f.suffix.lower() in
                  {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}])
    print(f"\n  [{cls}]  originals: {n_orig}", end=" → ")

    n_final = augment_class_folder(src, dst, TARGET_PER_CLASS)
    print(f"total after augmentation: {n_final}")
    total_generated += n_final

print(f"\n✅ Augmentation complete.  Grand total: {total_generated} images")

In [ ]:
# ============================================================
# CELL 6: Stratified Train / Validation Split
# ============================================================

def stratified_split(src_root: Path, dst_root: Path,
                     val_frac: float, test_frac: float, seed: int = SEED):
    random.seed(seed)
    summary = {}

    for cls_dir in sorted(src_root.iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name
        all_imgs = sorted(cls_dir.glob("*.*"))
        random.shuffle(all_imgs)

        n_test   = int(len(all_imgs) * test_frac)
        n_val    = int(len(all_imgs) * val_frac)
        test_imgs = all_imgs[:n_test]
        val_imgs  = all_imgs[n_test:n_test + n_val]
        trn_imgs  = all_imgs[n_test + n_val:]

        for split, imgs in [("train", trn_imgs), ("val", val_imgs), ("test", test_imgs)]:
            out_dir = dst_root / split / cls_name
            out_dir.mkdir(parents=True, exist_ok=True)
            for p in imgs:
                shutil.copy(p, out_dir / p.name)

        summary[cls_name] = {
            "train": len(trn_imgs),
            "val":   len(val_imgs),
            "test":  len(test_imgs),
        }

    return summary


print("Splitting into train / val …")
split_info = stratified_split(AUGMENTED_PATH, FINAL_PATH, VAL_SPLIT, TEST_SPLIT)

print(f"\n{'Class':<25} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 55)
for cls, counts in split_info.items():
    print(f"{cls:<25} {counts['train']:>8} {counts['val']:>8} {counts['test']:>8}")

total_train = sum(v["train"] for v in split_info.values())
total_val   = sum(v["val"]   for v in split_info.values())
total_test  = sum(v["test"]  for v in split_info.values())
print("-" * 55)
print(f"{'TOTAL':<25} {total_train:>8} {total_val:>8} {total_test:>8}")

In [ ]:
# ============================================================
# CELL 7: DataLoaders (NO augmentation — already done offline)
# ============================================================

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.ImageFolder(FINAL_PATH / "train",
                                     transform=train_transform)
val_dataset   = datasets.ImageFolder(FINAL_PATH / "val",
                                     transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

idx_to_cls = {v: k for k, v in train_dataset.class_to_idx.items()}

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Class mapping      : {train_dataset.class_to_idx}")

In [ ]:
# ============================================================
# CELL 8: EfficientNetB0 with Custom Head (5-class)
# ============================================================

def build_model(num_classes: int, device: torch.device) -> nn.Module:
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model   = efficientnet_b0(weights=weights)

    # Freeze entire backbone initially
    for param in model.parameters():
        param.requires_grad = False

    # Custom classification head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )

    return model.to(device)


model = build_model(NUM_CLASSES, DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable (head)    : {trainable_params:,}")

In [ ]:
# ============================================================
# CELL 9: Training Utilities
# ============================================================

class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4, restore_best=True):
        self.patience     = patience
        self.min_delta    = min_delta
        self.restore_best = restore_best
        self.best_loss    = float("inf")
        self.counter      = 0
        self.best_weights = None
        self.stop         = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss    = val_loss
            self.counter      = 0
            if self.restore_best:
                self.best_weights = {k: v.cpu().clone()
                                     for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                if self.restore_best and self.best_weights:
                    model.load_state_dict(
                        {k: v.to(DEVICE)
                         for k, v in self.best_weights.items()}
                    )


def run_epoch(model, loader, criterion, optimizer, phase="train"):
    is_train = (phase == "train")
    model.train() if is_train else model.eval()

    running_loss, running_correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds            = outputs.argmax(dim=1)
            running_loss    += loss.item() * imgs.size(0)
            running_correct += (preds == labels).sum().item()
            total           += imgs.size(0)

    return running_loss / total, running_correct / total


def unfreeze_top_layers(model, unfreeze_pct=0.30):
    feature_layers  = list(model.features.children())
    n_unlock        = max(1, int(len(feature_layers) * unfreeze_pct))
    layers_to_unlock = feature_layers[-n_unlock:]

    for layer in layers_to_unlock:
        for param in layer.parameters():
            param.requires_grad = True

    unlocked = sum(p.numel() for p in model.parameters()
                   if p.requires_grad)
    total    = sum(p.numel() for p in model.parameters())
    print(f"  Unlocked last {n_unlock}/{len(feature_layers)} feature blocks")
    print(f"  Trainable params: {unlocked:,} / {total:,} "
          f"({100*unlocked/total:.1f}%)")


print("✅ Training utilities ready.")

In [ ]:
# ============================================================
# CELL 10: Phase 1 — Train classifier head (backbone frozen)
# ============================================================

# Ordinal-aware label smoothing — adjacent grade errors penalized less
# Using standard CrossEntropy with smoothing; ordinal loss in Phase 2
criterion    = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler_p1  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=NUM_EPOCHS, eta_min=1e-6
)
early_stop_p1 = EarlyStopping(patience=PATIENCE, restore_best=True)

history_p1 = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   []}

print("=" * 60)
print("PHASE 1: HEAD TRAINING  (backbone frozen)")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(model, train_loader,
                                criterion, optimizer_p1, "train")
    vl_loss, vl_acc = run_epoch(model, val_loader,
                                criterion, None, "val")
    scheduler_p1.step()

    history_p1["train_loss"].append(tr_loss)
    history_p1["train_acc"].append(tr_acc)
    history_p1["val_loss"].append(vl_loss)
    history_p1["val_acc"].append(vl_acc)

    elapsed = time.time() - t0
    print(f"Ep {epoch:02d}/{NUM_EPOCHS}  "
          f"T-loss:{tr_loss:.4f}  T-acc:{tr_acc:.4f}  |  "
          f"V-loss:{vl_loss:.4f}  V-acc:{vl_acc:.4f}  "
          f"[{elapsed:.1f}s]")

    early_stop_p1(vl_loss, model)
    if early_stop_p1.stop:
        print(f"\n⏹  Early stopping at epoch {epoch} "
              f"(best val-loss: {early_stop_p1.best_loss:.4f})")
        break

print("\n✅ Phase 1 complete.")
torch.save(model.state_dict(),
           "/kaggle/working/quality_efficientnet_b0_phase1.pth")
print("   Checkpoint saved → quality_efficientnet_b0_phase1.pth")